### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [13]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] =os.getenv("GROQ_API_KEY")
llm = init_chat_model("qwen3:8b", model_provider="ollama")

llm
model = llm
response = model.invoke("Why do parrots talk?")

In [14]:
response

AIMessage(content="Parrots talk for a combination of biological, social, and evolutionary reasons. Here's a structured explanation:\n\n1. **Social Bonding and Communication**:  \n   Parrots are highly social animals, and mimicry plays a key role in their interactions. In the wild, they use a variety of calls to communicate with flock members. Mimicking human speech allows them to interact with humans, fostering bonds and reinforcing their role as companions. This behavior can be seen as an extension of their natural communication skills.\n\n2. **Learning and Exploration**:  \n   Parrots are intelligent and curious. Mimicking sounds is a form of play and learning, helping them explore their environment. They may experiment with vocalizations to understand their surroundings, much like how human children learn language through repetition and imitation.\n\n3. **Evolutionary Adaptations**:  \n   Parrots have evolved complex vocal systems, including a specialized syrinx (voice box), which a

In [17]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [19]:
response = model_with_tools.invoke("What's the weather like in Pune?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={} response_metadata={'model': 'qwen3:8b', 'created_at': '2026-06-27T17:58:40.3168912Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6188809500, 'load_duration': 665407000, 'prompt_eval_count': 138, 'prompt_eval_duration': 437623000, 'eval_count': 94, 'eval_duration': 5067572000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'} id='lc_run--019f0a3b-b98e-7b12-aea4-615237665603-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '20be9661-46a2-4503-bb9c-a7421169c863', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 138, 'output_tokens': 94, 'total_tokens': 232}
Tool: get_weather
Args: {'location': 'Pune'}


### Tool Execution Loops

In [20]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. Enjoy the nice weather! ☀️


In [21]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-06-27T18:02:04.0751479Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8217655200, 'load_duration': 676616300, 'prompt_eval_count': 137, 'prompt_eval_duration': 2441283000, 'eval_count': 94, 'eval_duration': 5080037000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019f0a3e-cd90-7321-b75b-566a42e0d31f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'ba17c76d-d15a-4ca9-91e6-04fad2caa397', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 94, 'total_tokens': 231}),
 ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='ba17c76d-d15a-4ca9-91e6-04fad2caa397')]